In [2]:
import MDAnalysis as mda
from numpy import *
import os
from pylab import *
import MDAnalysis.analysis.distances
import MDAnalysis.analysis.rms
from MDAnalysis.analysis import align
import glob
#import umap
import scipy.stats
import sklearn
import sklearn.decomposition
import sklearn.preprocessing
import pandas as pd
import seaborn as sns
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA
import mdtraj

/home/liam/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/chemfiles.py:59: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  MIN_CHEMFILES_VERSION = LooseVersion("0.9")


In [12]:
import os

########################################################
#############      FOR NOW EQPOINT IS 0   ##############
########################################################
EQPOINT=0

systemFolders = sorted(glob.glob("huNumbering/*t5a*/"))

systemgros=[]
systemtprs=[]
systemtrjs=[]
for i in range(len(systemFolders)):
    systemgros.append(sorted(glob.glob(systemFolders[i]+"*.gro")))
    systemtprs.append(sorted(glob.glob(systemFolders[i]+"*.tpr")))
    systemtrjs.append(sorted(glob.glob(systemFolders[i]+"*.xtc")))


    
    
threeColor=["#FE6100","#332288","#882255"]
colourScheme= threeColor
system_names = ["rhT5A","T5A","T5AR332P"]
systems=[]
for i in range(len(systemgros)):
    sub=[]
    for j in range(len(systemgros[i])):
        # When using TPRs, residues are indexed from 1; so we need to add in the first residue, 1 - 1 + first resid=first resid
        #firstres = mda.Universe(systemgros[i][j]).residues.resids[0]-1
        tu = mda.Universe(systemgros[i][j],systemtrjs[i][j])
        #tu.residues.resids +=firstres
                          
        sub.append(tu)
        
    systems.append(sub)

rhresids = arange(291,495)
huresids = systems[1][0].residues.resids
    
protein=[]
proteinstrings=[]
for i in range(len(systems)):
    sub = []
    sub2 = []
    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein"))
        sub2.append("protein")
        
    protein.append(sub)
    proteinstrings.append(sub2)


bodys=[]
bodystrings=[]
# rhesus v1 is 326-349
# Human v1 is 324-345
#huloopstring = "resid 324:345"
#rhloopstring = "resid 326:349"
combinedLoopString="resid 324:345"
#system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    sub2 = []
    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and not ("+combinedLoopString+")"))
        sub2.append("protein and not ("+combinedLoopString+")")
        
    bodys.append(sub)
    bodystrings.append(sub2)
    
    
v1s=[]
v1strings=[]
# rhesus v1 is 326-349
# Human v1 is 324-345
#huloopstring = "resid 324:345"
#rhloopstring = "resid 326:349"
combinedLoopString="resid 324:345"
#system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    sub2 = []

    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and "+combinedLoopString))
        sub2.append("protein and "+combinedLoopString)

        
    v1s.append(sub)        
    v1strings.append(sub2)
                   

In [21]:
systems[0][0].residues.resids

array([ 290,  291,  292,  293,  294,  295,  296,  297,  298,  299,  300,
        301,  302,  303,  304,  305,  306,  307,  308,  309,  310,  311,
        312,  313,  314,  315,  316,  317,  318,  319,  320,  321,  322,
        323,  324,  325,  326,  327,  328,  329,  330,  331,  332,  333,
        334,  335,  336, 1339, 1340,  337,  338,  339,  340,  341,  342,
        343,  344,  345,  346,  347,  348,  349,  350,  351,  352,  353,
        354,  355,  356,  357,  358,  359,  360,  361,  362,  363,  364,
        365,  366,  367,  368,  369,  370,  371,  372,  373,  374,  375,
        376,  377,  378,  379,  380,  381,  382,  383,  384,  385,  386,
        387,  388,  389,  390,  391,  392,  393,  394,  395,  396,  397,
        398,  399,  400,  401,  402,  403,  404,  405,  406,  407,  408,
        409,  410,  411,  412,  413,  414,  415,  416,  417,  418,  419,
        420,  421,  422,  423,  424,  425,  426,  427,  428,  429,  430,
        431,  432,  433,  434,  435,  436,  437,  4

In [37]:
r=systems[0][0]
h=systems[1][0]
diffRes=[]
for i in range(len(systems[0][0].residues.resids)):
    rn = r.select_atoms(f"resid {systems[0][0].residues.resids[i]}").residues.resnames[0]
    try:hn = h.select_atoms(f"resid {systems[0][0].residues.resids[i]}").residues.resnames[0]
    except: pass
    if rn != hn:
        print(r.residues.resids[i], r.residues.resnames[i], h.residues.resnames[i])
        diffRes.append(r.residues.resids[i])

294 ALA VAL
303 LEU VAL
310 HIS CYS
314 ALA SER
323 ARG PRO
324 ASN LYS
328 MET ILE
330 GLN GLY
332 PRO ARG
335 LEU ARG
336 PHE TYR
1339 THR GLN
1340 PHE THR
337 PRO PHE
338 SER VAL
339 LEU ASN
340 THR PHE
348 VAL GLY
369 SER TRP
381 SER ALA
385 TYR ILE
389 GLN GLU
405 GLN GLY
410 TYR ALA
412 VAL GLN
416 GLY PHE
418 SER THR
422 PHE PRO
423 ALA PHE
442 VAL TYR
466 GLN SER
471 LYS VAL
483 THR PRO


In [38]:
mystr=""
for i in range(len(diffRes)):
    mystr+=f"resid {diffRes[i]} or "

In [41]:
mystr

'resid 294 or resid 303 or resid 310 or resid 314 or resid 323 or resid 324 or resid 328 or resid 330 or resid 332 or resid 335 or resid 336 or resid 1339 or resid 1340 or resid 337 or resid 338 or resid 339 or resid 340 or resid 348 or resid 369 or resid 381 or resid 385 or resid 389 or resid 405 or resid 410 or resid 412 or resid 416 or resid 418 or resid 422 or resid 423 or resid 442 or resid 466 or resid 471 or resid 483 or '